# AgentCore Runtime: From Local Agent to Deployed Endpoint

This notebook walks through the full lifecycle of deploying a Strands agent to Amazon Bedrock AgentCore Runtime:

1. Write the agent code with `BedrockAgentCoreApp`
2. Test it locally
3. Configure the Runtime deployment
4. Launch to AgentCore
5. Invoke the deployed agent (Starter Toolkit + boto3)
6. Manage sessions
7. Clean up

**Prerequisites:**
- AWS account with Bedrock model access enabled
- AWS credentials configured (`aws configure` or environment variables)
- Python 3.10+

> This notebook accompanies Chapter 6 of *Agentic AI on AWS*.

## Step 0: Install Dependencies

In [ ]:
%pip install bedrock-agentcore strands-agents strands-agents-tools bedrock-agentcore-starter-toolkit boto3 --quiet

## Step 1: Write the Agent Code

This is the same document analysis agent from the chapter. The only additions over a plain Strands agent are:
- `BedrockAgentCoreApp` import and instance
- `@app.entrypoint` decorator on the handler
- `app.run()` in the `__main__` block

We write it to a file so the Starter Toolkit can package and deploy it.

In [ ]:
%%writefile agent.py
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models.bedrock import BedrockModel

app = BedrockAgentCoreApp()


@tool
def extract_clauses(text: str) -> dict:
    """Extract key clauses from contract text."""
    # In production, this would use NLP or a specialized model.
    # For this example, we return a structured placeholder.
    return {"clauses": ["termination", "liability", "indemnification"]}


model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
agent = Agent(
    model=model,
    tools=[extract_clauses],
    system_prompt="You are a document analysis assistant. Extract and summarize key contract clauses.",
)


@app.entrypoint
def handle(payload):
    response = agent(payload.get("prompt"))
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()

Create a `requirements.txt` for the deployment package:

In [ ]:
%%writefile requirements.txt
bedrock-agentcore
strands-agents
strands-agents-tools

## Step 2: Test Locally

Before deploying anywhere, verify the agent works on your machine. Run the following in a **separate terminal**:

```bash
python agent.py
```

This starts a local HTTP server on port 8080 with two endpoints:
- `POST /invocations` — send requests to your agent
- `GET /ping` — health check

Test it with curl:

```bash
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -d '{"prompt": "Extract the key clauses from this contract: The vendor shall be liable for..." }'
```

Once you have confirmed it works locally, stop the server (Ctrl+C) and continue with deployment.

## Step 3: Configure the Runtime Deployment

The Starter Toolkit's `Runtime` class handles the full deployment lifecycle. The `configure` step:
- Generates a Dockerfile from your code
- Creates an IAM execution role with the permissions your agent needs
- Sets up an ECR repository for the container image

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

region = Session().region_name
print(f"Deploying to region: {region}")

runtime = Runtime()

runtime.configure(
    entrypoint="agent.py",
    requirements_file="requirements.txt",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name="doc_analysis_agent",
)

## Step 4: Launch to AgentCore

This builds the container image, pushes it to ECR, and creates the AgentCore Runtime endpoint. The first deployment takes a few minutes while it packages everything. Subsequent updates are faster because the toolkit reuses cached dependencies.

In [ ]:
launch_result = runtime.launch()

print(f"Agent ID:  {launch_result.agent_id}")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"ECR URI:   {launch_result.ecr_uri}")

## Step 5: Wait for the Endpoint to Be Ready

The Runtime endpoint goes through several states before it can accept requests. Poll until it reaches `READY`.

In [ ]:
import time

status = runtime.status().endpoint["status"]
print(f"Status: {status}")

while status not in ["READY", "CREATE_FAILED"]:
    time.sleep(10)
    status = runtime.status().endpoint["status"]
    print(f"Status: {status}")

if status == "READY":
    print("\nEndpoint is ready. You can now invoke the agent.")
else:
    print("\nDeployment failed. Check the AWS console for details.")

## Step 6: Invoke with the Starter Toolkit

The simplest way to call your deployed agent — the Starter Toolkit wraps the API call for you.

In [ ]:
response = runtime.invoke({"prompt": "Analyze this contract clause: The vendor shall be liable for direct damages only, excluding consequential, incidental, or punitive damages."})
print(response["response"][0])

## Step 7: Invoke with boto3

In production, you would call the deployed agent from other AWS services using boto3 directly. This is how a Lambda function, an ECS service, or a Step Functions workflow would invoke your agent.

In [ ]:
import boto3
import json
import uuid

client = boto3.client("bedrock-agentcore", region_name=region)

response = client.invoke_agent_runtime(
    agentRuntimeArn=launch_result.agent_arn,
    runtimeSessionId=str(uuid.uuid4()),
    qualifier="DEFAULT",
    payload=json.dumps(
        {"prompt": "Summarize the liability and indemnification clauses in this contract."}
    ),
)

session_id = response.get("runtimeSessionId")
print(f"Session ID: {session_id}")

# Read the actual agent response from the chunked streaming body
content = []
for chunk in response.get("response", []):
    content.append(chunk.decode("utf-8"))
response_body = json.loads("".join(content))
print(f"\nAgent response:\n{response_body}")


## Step 8: Stop the Session

Each session runs in its own microVM. When you are done with a conversation, stop the session explicitly to release compute resources and avoid unnecessary charges. If you do not stop it manually, the session will terminate after the idle timeout (default: 15 minutes).

In [ ]:
if session_id:
    client.stop_runtime_session(
        agentRuntimeArn=launch_result.agent_arn,
        runtimeSessionId=session_id,
        qualifier="DEFAULT",
    )
    print(f"Session {session_id} stopped.")
else:
    print("No session to stop (invoke may not have returned a session ID).")

## Step 9: Clean Up

Delete the Runtime endpoint and the ECR repository to avoid ongoing charges. This removes all deployed resources.

In [ ]:
agentcore_control = boto3.client("bedrock-agentcore-control", region_name=region)
ecr = boto3.client("ecr", region_name=region)

# Delete the AgentCore Runtime
try:
    agentcore_control.delete_agent_runtime(agentRuntimeId=launch_result.agent_id)
    print(f"Deleted AgentCore Runtime: {launch_result.agent_id}")
except Exception as e:
    print(f"Error deleting runtime: {e}")

# Delete the ECR repository
try:
    repo_name = launch_result.ecr_uri.split("/")[1]
    ecr.delete_repository(repositoryName=repo_name, force=True)
    print(f"Deleted ECR repository: {repo_name}")
except Exception as e:
    print(f"Error deleting ECR repo: {e}")

---

## What You Just Did

You took a standard Strands agent, added four lines of AgentCore boilerplate, and deployed it to a managed runtime with session isolation, health checks, and auto-scaling — without writing a Dockerfile, a SAM template, or a load balancer configuration.

The agent code itself never changed. The same `Agent`, `tool`, and `BedrockModel` you have been using since Chapter 2 work identically here.

**Next steps:**
- Try modifying `agent.py` with additional tools and re-running `runtime.launch()` to update the deployment
- Explore the [AgentCore samples repository](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-tutorials/01-AgentCore-runtime) for LangGraph, OpenAI, and advanced lifecycle examples
- Continue to Chapter 7 for observability, guardrails, and cost optimization